In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "processed" / "litsearch"


In [2]:
from src.rerank.fine import run_fine_reranking


In [3]:
import json

def dummy_fine_llm(prompt: str) -> str:
    # для отладки — возвращаем пустой список
    return json.dumps([])


In [4]:
run_fine_reranking(
    coarse_path=DATA_DIR / "coarse_rerank.jsonl",
    queries_path=DATA_DIR / "queries.jsonl",
    corpus_path=DATA_DIR / "corpus.jsonl",
    output_path=DATA_DIR / "fine_rerank.jsonl",
    llm_call_fn=dummy_fine_llm,
)


In [5]:
!head -n 3 ../data/processed/litsearch/fine_rerank.jsonl | jq .

{
  "query_id": 0,
  "final_ranked_doc_ids": []
}
{
  "query_id": 1,
  "final_ranked_doc_ids": []
}
{
  "query_id": 2,
  "final_ranked_doc_ids": []
}


In [6]:
import json

def count_lines(path):
    with open(path, "r", encoding="utf-8") as f:
        return sum(1 for _ in f)

print("queries:", count_lines(DATA_DIR / "queries.jsonl"))
print("candidates:", count_lines(DATA_DIR / "candidates.jsonl"))
print("coarse:", count_lines(DATA_DIR / "coarse_rerank.jsonl"))
print("fine:", count_lines(DATA_DIR / "fine_rerank.jsonl"))


queries: 597
candidates: 597
coarse: 597
fine: 597


In [9]:
QUERY_ID = 0

with open(DATA_DIR / "queries.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        q = json.loads(line)
        if q["qid"] == QUERY_ID:
            query_text = q["text"]
            break

query_text


'Are there any research papers on methods to compress large-scale language models using task-agnostic knowledge distillation techniques?'

In [10]:
with open(DATA_DIR / "candidates.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        if item["query_id"] == QUERY_ID:
            bm25_candidates = item["candidates"]
            break

len(bm25_candidates), bm25_candidates[:5]

(200,
 [{'doc_id': 221995575, 'score': 37.03681734794637},
  {'doc_id': 259137871, 'score': 35.94452799168773},
  {'doc_id': 256900817, 'score': 35.15916143210283},
  {'doc_id': 215238664, 'score': 33.40335362994966},
  {'doc_id': 218502458, 'score': 33.185204149058876}])

In [11]:
with open(DATA_DIR / "coarse_rerank.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        if item["query_id"] == QUERY_ID:
            coarse_ranked = item["ranked_doc_ids"]
            break

len(coarse_ranked), coarse_ranked[:10]


(0, [])

In [12]:
with open(DATA_DIR / "fine_rerank.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        if item["query_id"] == QUERY_ID:
            fine_ranked = item["final_ranked_doc_ids"]
            break

len(fine_ranked), fine_ranked[:10]


(0, [])

In [13]:
# загрузим корпус
corpus = {}
with open(DATA_DIR / "corpus.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        doc = json.loads(line)
        doc_id = doc.get("doc_id") or doc.get("corpusid")
        corpus[doc_id] = doc

def show_docs(doc_ids, title):
    print("=" * 80)
    print(title)
    print("=" * 80)
    for i, doc_id in enumerate(doc_ids[:5], 1):
        doc = corpus.get(doc_id, {})
        print(f"\n#{i}")
        print("doc_id:", doc_id)
        print("title:", doc.get("title", ""))
        print("abstract:", doc.get("abstract", "")[:500])


In [14]:
show_docs([c["doc_id"] for c in bm25_candidates], "BM25 top-5")


BM25 top-5

#1
doc_id: 221995575
title: Contrastive Distillation on Intermediate Representations for Language Model Compression
abstract: Existing language model compression methods mostly use a simple L 2 loss to distill knowledge in the intermediate representations of a large BERT model to a smaller one. Although widely used, this objective by design assumes that all the dimensions of hidden representations are independent, failing to capture important structural knowledge in the intermediate layers of the teacher network. To achieve better distillation efficacy, we propose Contrastive Distillation on Intermediate Representation

#2
doc_id: 259137871
title: GKD: A General Knowledge Distillation Framework for Large-scale Pre-trained Language Model
abstract: Currently, the reduction in the parameter scale of large-scale pre-trained language models (PLMs) through knowledge distillation has greatly facilitated their widespread deployment on various devices. However, the deployment of kno

In [15]:
show_docs(coarse_ranked, "Coarse rerank top-5")


Coarse rerank top-5


In [16]:
show_docs(fine_ranked, "Fine rerank top-5")


Fine rerank top-5
